# Sleep-EDF Dataset Building & Packing Pipeline

This script serves as the preprocessing and data-packing pipeline, converting raw Sleep-EDF pickle files into a unified, high-performance **PyTorch Tensor dataset (`full_dataset.pt`)**.

---

### Key Workflow Steps

1. **[1/4] Directory Scanning**: Loads file paths and checks target directories based on parameters defined in `config.json`.
2. **[2/4] Bulk Data Extraction**: Reads raw pickle (`.pkl`) files containing biosignals and sleep stage labels, mapping stage strings (`W`, `1`, `2`, `3`, `4`, `R`) to standard class indices (`0` to `4`).
3. **[3/4] Tensor Conversion & Reshaping**: Merges accumulated raw signal lists into a single consolidated PyTorch tensor with the shape `[Batch, Channels=7, Length=3000]`.
4. **[4/4] Binary Dataset Serialization**: Dumps the integrated tensor dictionary (`signals` and `labels`) into `full_dataset.pt` for high-speed, zero-overhead memory-mapped loading during model training.

In [ ]:
import os
import json
import datetime
import time
import torch
import pickle
from tqdm import tqdm
import random
from collections import defaultdict

# ---------------------------------------------------------------------
# Load configuration from config.json
# ---------------------------------------------------------------------
CONFIG_PATH = "config.json"

if os.path.exists(CONFIG_PATH):
    with open(CONFIG_PATH, "r", encoding="utf-8") as f:
        config = json.load(f)
    print(f"[Config Loaded] Successfully loaded settings from '{CONFIG_PATH}'.")
else:
    # Fallback default configuration if config.json does not exist
    config = {
        "TEMP_SAVE_DIR": "./pyhealth_temp",
        "TARGET_COUNT_PER_CLASS": 10000
    }
    print(f"[Config Warning] '{CONFIG_PATH}' not found. Using default paths.")

TEMP_SAVE_DIR = config.get("TEMP_SAVE_DIR", "./pyhealth_temp")
TARGET_COUNT_PER_CLASS = config.get("TARGET_COUNT_PER_CLASS", 10000)

print(f"[Packing Debug Mode] Script start time: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[Target Directory] {TEMP_SAVE_DIR}")

# =====================================================================
# [1/4] Scan target directory for files
# =====================================================================
print(f"\n[1/4] {datetime.datetime.now().strftime('%H:%M:%S')} | Starting disk file scan with os.listdir()...")
start_scan = time.time()

if not os.path.exists(TEMP_SAVE_DIR):
    os.makedirs(TEMP_SAVE_DIR, exist_ok=True)

file_names = [f for f in os.listdir(TEMP_SAVE_DIR) if f.endswith(".pkl")]
end_scan = time.time()
print(f"[1/4 Completed] Elapsed time: {end_scan - start_scan:.2f}s | Secured {len(file_names)} pickle file paths.")

label_map = {'W': 0, '1': 1, '2': 2, '3': 3, '4': 3, 'R': 4}
all_signals = []
all_labels = []

# =====================================================================
# [2/4] Bulk load Sleep-EDF pickle dataset
# =====================================================================
print(f"\n[2/4] {datetime.datetime.now().strftime('%H:%M:%S')} | Entering bulk pickle data reading loop...")
loop_start_time = time.time()
checkpoint_time = time.time()

report_interval = 10000  # Report progress every 10,000 files

for idx, f_name in enumerate(tqdm(file_names, desc="Packing raw pickle files")):
    pkl_path = os.path.join(TEMP_SAVE_DIR, f_name)
    try:
        with open(pkl_path, 'rb') as f:
            actual_data = pickle.load(f)

        if isinstance(actual_data, dict):
            signal_data = actual_data.get("signal", None)
            label_data = str(actual_data.get("label", "W"))
        else:
            signal_data = actual_data
            label_data = "W"

        if signal_data is None:
            continue

        all_signals.append(signal_data)
        all_labels.append(label_map.get(label_data, 0))
    except Exception as e:
        continue

    if (idx + 1) % report_interval == 0:
        now_time = time.time()
        interval_duration = now_time - checkpoint_time
        avg_speed = interval_duration / report_interval

        print(f"[{idx + 1}/{len(file_names)}] {datetime.datetime.now().strftime('%H:%M:%S')} | "
              f"Last {report_interval} samples: {interval_duration:.2f}s ({avg_speed*1000:.2f}ms/sample) | "
              f"Accumulated samples: {len(all_signals)}")
        checkpoint_time = now_time

print(f"[2/4 Completed] Successfully loaded all files! Total loop time: {time.time() - loop_start_time:.2f}s")

# =====================================================================
# [3/4] Convert Python lists to PyTorch Tensors
# =====================================================================
print(f"\n[3/4] {datetime.datetime.now().strftime('%H:%M:%S')} | Converting Python list structure into a single PyTorch Tensor...")

tensor_convert_start = time.time()

# Reshape into [Batch, Channels=7, Length=3000]
signals_tensor = torch.tensor(all_signals, dtype=torch.float32).view(-1, 7, 3000)
labels_tensor = torch.tensor(all_labels, dtype=torch.long)

print(f"[3/4 Completed] Tensor merging passed! (Conversion time: {time.time() - tensor_convert_start:.2f}s)")
print(f"Final Tensor Shapes: Signals {signals_tensor.shape} | Labels {labels_tensor.shape}")

# =====================================================================
# [4/4] Dump integrated tensor dataset to disk
# =====================================================================
output_pt_path = os.path.join(TEMP_SAVE_DIR, "full_dataset.pt")
print(f"\n[4/4] {datetime.datetime.now().strftime('%H:%M:%S')} | Saving full tensor dataset to '{output_pt_path}'...")
save_start = time.time()

torch.save({
    "signals": signals_tensor,
    "labels": labels_tensor
}, output_pt_path)

print(f"\n[FINAL COMPLETE] {datetime.datetime.now().strftime('%H:%M:%S')} | Integrated PyTorch tensor dataset build successful!")
print(f"Disk I/O Save Time: {time.time() - save_start:.2f}s")